# 10 — RQ3: Do market indicators improve on register-only forecasts?

**Question.** Do rental-market indicators improve forecast accuracy beyond the register's own history alone? Reuses the RQ2 walk-forward engine and adds a register+market forecaster; the naive baseline is kept so we can also say whether the fuller model is *useful* (beats naive), not only whether market data *helps* (beats register-only). Primary horizon = 1 quarter.

### 1. Dependencies

In [1]:
!pip -q install scipy matplotlib  # no-op if present

### 2. Mount Drive, enter the repo, pull first

In [2]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/capstone-project
!git pull --no-edit

Mounted at /content/drive
/content/drive/MyDrive/capstone-project
Already up to date.


### 3. Author the RQ3 module
Creates/overwrites `src/rq3.py`.

In [3]:
%%writefile src/rq3.py
"""
rq3.py — RQ3 incremental value of rental-market indicators over register-only forecasts.

Question
--------
Do rental-market indicators improve forecast accuracy beyond the register's own
history alone? (Rung 2 vs Rung 1.) The naive baseline is also reported, so we can
separately say whether the fuller model is *useful* (beats naive), not only whether
market data *helps* (beats register-only).

Design (settled in supervision)
-------------------------------
  models      : rw_flat  — naive random walk (carry last level; the usefulness bar)
                reg      — register-only AR(1) on the change, TA effects, no time effects
                mkt      — reg PLUS the four RQ1 market signals at lag 1
  market set  : rent_growth_lag1, net_bond_flow_lag1, dwelling_units_lag1,
                multi_unit_share_lag1  (same features as RQ1)
  primary     : horizon 1 — market lag-1 values are genuinely known at forecast time.
                Horizons 2-4 are SECONDARY and caveated: multi-step needs future market
                values, so the last known market state is carried forward (stale).
  validation  : expanding-window walk-forward, first origin t0 (~q20), all TAs.
  metrics     : out-of-sample RMSE / MAE per horizon; skill vs naive; incremental
                skill of mkt over reg; and a Diebold-Mariano test (reg vs mkt)
                CLUSTERED BY TARGET QUARTER to respect cross-sectional dependence.

Framing is prediction, never causation.
"""
import numpy as np
import pandas as pd
from scipy import stats

import config

LEVEL, CHANGE = "demand_rate", "demand_rate_change"
ENTITY, TIME = "ta_key", "q"
MARKET = ["rent_growth_lag1", "net_bond_flow_lag1",
          "dwelling_units_lag1", "multi_unit_share_lag1"]


def load_panel():
    return pd.read_parquet(config.DATA_PROCESSED / "panel.parquet")


def make_grids(panel):
    qs = sorted(panel[TIME].unique())
    qidx = {q: i for i, q in enumerate(qs)}
    p = panel.copy()
    p["ti"] = p[TIME].map(qidx)
    L = p.pivot(index="ti", columns=ENTITY, values=LEVEL).sort_index()
    C = p.pivot(index="ti", columns=ENTITY, values=CHANGE).sort_index()
    M = {c: p.pivot(index="ti", columns=ENTITY, values=c).sort_index() for c in MARKET}
    return L, C, M, qs


# ── fixed-effects AR(1) (+ optional market lag-1 exog), within-TA demeaned ──
def fit(Ctrain, M, tas, exog):
    rows = []
    for i in tas:
        s = Ctrain[i].dropna()
        sidx = set(s.index)
        for t in s.index:
            if (t - 1) in sidx:
                mk = [M[c][i][t] for c in exog]
                if any(pd.isna(mk)):
                    continue
                rows.append([i, s[t], s[t - 1]] + mk)
    if len(rows) < 50:
        return None
    feats = ["x0"] + list(exog)
    df = pd.DataFrame(rows, columns=["i", "y", "x0"] + list(exog))
    for c in ["y"] + feats:
        df[c + "_d"] = df[c] - df.groupby("i")[c].transform("mean")
    b, *_ = np.linalg.lstsq(df[[c + "_d" for c in feats]].values, df["y_d"].values, rcond=None)
    ci = {i: g["y"].mean() - sum(b[k] * g[feats[k]].mean() for k in range(len(feats)))
          for i, g in df.groupby("i")}
    return b, ci


def fcast(i, t, L, C, M, fitres, exog, h):
    b, ci = fitres
    if i not in ci or np.isnan(C[i][t]):
        return np.nan
    mk = [M[c][i][t + 1] for c in exog]     # market known at origin t (exact for h=1; carried forward for h>1)
    if any(pd.isna(mk)):
        return np.nan
    prev, ds = C[i][t], []
    for _ in range(h):
        d = ci[i] + b[0] * prev + sum(b[k + 1] * mk[k] for k in range(len(exog)))
        ds.append(d)
        prev = d
    return L[i][t] + np.sum(ds)


# ───────────────────────── walk-forward + scoring ─────────────────────────
def walk_forward(L, C, M, t0=20, horizons=(1, 2, 3, 4)):
    T = L.shape[0]
    tas = list(L.columns)
    models = ["rw_flat", "reg", "mkt"]
    err = {m: {h: {"se": [], "ae": []} for h in horizons} for m in models}
    dm = {h: [] for h in horizons}     # (target_quarter, e_reg^2 - e_mkt^2)
    for t in range(t0, T):
        fr = fit(C.loc[:t], M, tas, [])
        fm = fit(C.loc[:t], M, tas, MARKET)
        for i in tas:
            Lt = L[i][t]
            if np.isnan(Lt):
                continue
            for h in horizons:
                if t + h >= T:
                    continue
                a = L[i][t + h]
                if np.isnan(a):
                    continue
                fc = {"rw_flat": Lt,
                      "reg": fcast(i, t, L, C, M, fr, [], h) if fr else np.nan,
                      "mkt": fcast(i, t, L, C, M, fm, MARKET, h) if fm else np.nan}
                for m in models:
                    if not np.isnan(fc[m]):
                        err[m][h]["se"].append((a - fc[m]) ** 2)
                        err[m][h]["ae"].append(abs(a - fc[m]))
                if not np.isnan(fc["reg"]) and not np.isnan(fc["mkt"]):
                    dm[h].append((t + h, (a - fc["reg"]) ** 2 - (a - fc["mkt"]) ** 2))
    return err, models, dm


def diebold_mariano(dm_h):
    """Clustered-by-quarter DM: H0 reg and mkt have equal forecast accuracy.
    mean loss diff = mean(e_reg^2 - e_mkt^2); >0 => mkt better."""
    d = pd.DataFrame(dm_h, columns=["q", "d"])
    dbar, N = d["d"].mean(), len(d)
    v = sum((g["d"].sum() - len(g) * dbar) ** 2 for _, g in d.groupby("q")) / N ** 2
    DM = dbar / np.sqrt(v)
    return {"DM": DM, "p_value": 2 * (1 - stats.norm.cdf(abs(DM))),
            "mean_loss_diff": dbar, "n": N}


def score(err, models, dm, horizons=(1, 2, 3, 4)):
    rmse = lambda m, h: np.sqrt(np.mean(err[m][h]["se"])) if err[m][h]["se"] else np.nan
    mae = lambda m, h: np.mean(err[m][h]["ae"]) if err[m][h]["ae"] else np.nan
    idx = [f"h{h}" for h in horizons]
    R = pd.DataFrame({m: [rmse(m, h) for h in horizons] for m in models}, index=idx).T
    Ma = pd.DataFrame({m: [mae(m, h) for h in horizons] for m in models}, index=idx).T
    skill_naive = pd.DataFrame({m: [1 - rmse(m, h) / rmse("rw_flat", h) for h in horizons]
                                for m in ["reg", "mkt"]}, index=idx).T
    incr = pd.Series([1 - rmse("mkt", h) / rmse("reg", h) for h in horizons], index=idx,
                     name="mkt_vs_reg")
    dm_res = {f"h{h}": diebold_mariano(dm[h]) for h in horizons}
    return {"rmse": R.round(4), "mae": Ma.round(4),
            "skill_vs_naive": skill_naive.round(3),
            "incremental_skill_mkt_vs_reg": incr.round(3), "dm": dm_res}


def run(panel=None, t0=20):
    panel = load_panel() if panel is None else panel
    L, C, M, qs = make_grids(panel)
    err, models, dm = walk_forward(L, C, M, t0=t0)
    out = {"t0": t0, "origin_quarter": qs[t0], "err": err, "models": models, "dm_raw": dm}
    out.update(score(err, models, dm))
    return out


def print_report(out=None):
    out = run() if out is None else out
    print(f"RQ3 — incremental value of market signals. Walk-forward, first forecast {out['origin_quarter']}.")
    print("Models: rw_flat (naive), reg (register-only), mkt (register + 4 market signals).\n")
    print("RMSE by horizon:"); print(out["rmse"].to_string(), "\n")
    print("Skill vs naive (both should be checked for absolute usefulness):")
    print(out["skill_vs_naive"].to_string(), "\n")
    print("RQ3 direct — incremental skill of mkt over reg (1 - RMSE_mkt/RMSE_reg; >0 = market helps):")
    print(out["incremental_skill_mkt_vs_reg"].to_string(), "\n")
    print("Diebold-Mariano (reg vs mkt), clustered by target quarter  [h1 = primary]:")
    for h, r in out["dm"].items():
        print(f"  {h}: DM = {r['DM']:+.2f}  p = {r['p_value']:.3f}  "
              f"(mean loss diff = {r['mean_loss_diff']:+.4f}; >0 => market better; n = {r['n']})")
    return out


Overwriting src/rq3.py


### 4. Run RQ3 and display the result

In [4]:
import sys, importlib, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, 'src')
import rq3; importlib.reload(rq3)

out = rq3.print_report()

RQ3 — incremental value of market signals. Walk-forward, first forecast 2021Q1.
Models: rw_flat (naive), reg (register-only), mkt (register + 4 market signals).

RMSE by horizon:
             h1      h2      h3      h4
rw_flat  0.4601  0.6717  0.8415  0.9641
reg      0.5073  0.8137  1.1067  1.3883
mkt      0.4961  0.7927  1.0682  1.3415 

Skill vs naive (both should be checked for absolute usefulness):
        h1     h2     h3     h4
reg -0.103 -0.211 -0.315 -0.440
mkt -0.078 -0.180 -0.270 -0.391 

RQ3 direct — incremental skill of mkt over reg (1 - RMSE_mkt/RMSE_reg; >0 = market helps):
h1    0.022
h2    0.026
h3    0.035
h4    0.034 

Diebold-Mariano (reg vs mkt), clustered by target quarter  [h1 = primary]:
  h1: DM = +1.46  p = 0.145  (mean loss diff = +0.0029; >0 => market better; n = 1118)
  h2: DM = +1.71  p = 0.088  (mean loss diff = +0.0102; >0 => market better; n = 1057)
  h3: DM = +1.68  p = 0.094  (mean loss diff = +0.0251; >0 => market better; n = 995)
  h4: DM = +1.75  p 

### 5. Inspect individual pieces (optional)

In [ ]:
out['rmse']

In [ ]:
out['incremental_skill_mkt_vs_reg']  # RQ3 direct answer

In [ ]:
out['dm']['h1']  # Diebold-Mariano at the primary horizon

### 6. Comparison figure

In [ ]:
### 7. Comparison figure (saves to outputs/)
import numpy as np, matplotlib.pyplot as plt
R, SK = out['rmse'], out['skill_vs_naive']
incr, dm = out['incremental_skill_mkt_vs_reg'], out['dm']
H=['h1','h2','h3','h4']; x=np.arange(len(H)); w=0.26
fig, ax = plt.subplots(1,2, figsize=(12.5,4.8))
ax[0].bar(x-w, R.loc['rw_flat'], w, label='naive (no-change)', color='#27ae60')
ax[0].bar(x,   R.loc['reg'],     w, label='register-only',     color='#2c3e50')
ax[0].bar(x+w, R.loc['mkt'],     w, label='register + market', color='#c0392b')
ax[0].set_xticks(x); ax[0].set_xticklabels(['1q','2q','3q','4q'])
ax[0].set_xlabel('forecast horizon'); ax[0].set_ylabel('RMSE (per 1,000)')
ax[0].set_title('A. Naive wins at every horizon; market barely shifts register-only', fontsize=9.5, loc='left')
ax[0].legend(fontsize=8, frameon=False)
ax[1].axhline(0, color='gray', lw=.8)
ax[1].bar(x-w/2, SK.loc['reg'], w, label='register-only vs naive', color='#2c3e50')
ax[1].bar(x+w/2, SK.loc['mkt'], w, label='register+market vs naive', color='#c0392b')
ax[1].set_xticks(x); ax[1].set_xticklabels(['1q','2q','3q','4q'])
ax[1].set_xlabel('forecast horizon'); ax[1].set_ylabel('skill vs naive  (>0 beats naive)')
ax[1].set_title('B. Both lose to naive; market closes only part of the gap', fontsize=9.5, loc='left')
ax[1].legend(fontsize=8, frameon=False, loc='lower left')
for xi,h in zip(x,H):
    ax[1].annotate(f"+{incr[h]*100:.1f}%\nDM p={dm[h]['p_value']:.2f}",
                   xy=(xi, min(SK.loc['reg',h], SK.loc['mkt',h])-0.02),
                   ha='center', va='top', fontsize=7, color='#555')
plt.tight_layout()
plt.savefig('outputs/rq3_comparison.png', dpi=150, bbox_inches='tight'); plt.show()

### 7. Commit & push
**Save the notebook first** (Ctrl/Cmd-S).

In [ ]:
%cd /content/drive/MyDrive/capstone-project

from google.colab import userdata
tok = userdata.get('GH_TOKEN')

!git config user.name  'fashdeen'
!git config user.email 'fashdeen@yahoo.com'

!git add src/rq3.py notebooks/10_rq3.ipynb outputs/rq3_comparison.png
!git commit -m 'RQ3: incremental value of market signals (walk-forward + Diebold-Mariano)'
!git push https://{tok}@github.com/fashdeen/capstone-project.git